# Model Training - Energy Grid Outage Prediction

Trains a PySpark RandomForest classifier to predict grid outages.

Target: had_outage (1 = outage/surge/sag event occurred)
Reads from: gold_ml_features
Writes to: gold_ml_model_metrics table and Files/models/outage_prediction_rf

In [ ]:
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print('Imports OK')

In [ ]:
# Load engineered features
df = spark.read.table('gold_ml_features')
print(f'Feature rows: {df.count()}')
print(f'Columns: {df.columns}')

In [ ]:
# Clean nulls / NaN in numeric columns so the trainer never sees invalid values
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

df.groupBy('had_outage').count().show()

In [ ]:
# Auto-discover numeric feature columns (exclude label and leakage columns)
exclude = ('had_outage', 'event_count', 'total_affected', 'total_event_duration')
feature_cols = [
    c for c, dtype in df.dtypes
    if dtype in ('double', 'float', 'int', 'bigint', 'long') and c not in exclude
]
print(f'Features ({len(feature_cols)}): {feature_cols}')

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(df).select('features', col('had_outage').cast('double').alias('label'))
model_df = model_df.cache()
print(f'Model rows: {model_df.count()}')

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count()}, Test: {test_df.count()}')

In [ ]:
# Train RandomForest (built into PySpark, no SynapseML dependency)
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    predictionCol='prediction',
    rawPredictionCol='rawPrediction',
    probabilityCol='probability',
    numTrees=50,
    maxDepth=8,
    seed=42,
)
model = rf.fit(train_df)
print('RandomForest model trained')

In [ ]:
# Evaluate
predictions = model.transform(test_df)

auc = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy').evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1').evaluate(predictions)

print(f'AUC-ROC:  {auc:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print(f'F1:       {f1:.4f}')

In [ ]:
# Save evaluation metrics to a Gold table
metrics = spark.createDataFrame(
    [('energy-grid', 'outage-prediction', 'RandomForestClassifier',
      len(feature_cols), train_df.count(), test_df.count(),
      float(auc), float(accuracy), float(f1))],
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())

metrics.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_model_metrics')
print('Metrics saved to gold_ml_model_metrics')

In [ ]:
# Persist the trained model (overwrite so re-runs are idempotent)
model.write().overwrite().save('Files/models/outage_prediction_rf')
model_df.unpersist()
print('Model saved to Files/models/outage_prediction_rf')